# Step 0 — Download & Build Training Dataset

This notebook demonstrates how to:
1. Download a small subset of cpg0000-jump-pilot images from the Cell Painting Gallery
2. Build a MicroSplit training dataset from those images

For full-scale HPC runs, see `../cpg0000-jump-pilot/datasets.sh`.

**Requirements:**
```bash
pip install jump-portrait tifffile pandas numpy
```

## 1. Download images from Cell Painting Gallery

The [Cell Painting Gallery](https://github.com/broadinstitute/cellpainting-gallery) stores images on AWS S3.
`jump_portrait` provides a convenient Python API to fetch them.

We'll download a single well (A549, well B02, a few sites) to keep things fast.

In [ ]:
from pathlib import Path
import numpy as np

# Configuration
DOWNLOAD_DIR = Path('./cpg0000_sample_data')
DATASET_DIR  = Path('./cpg0000_training_dataset')
CELL_LINE    = 'A549'

# cpg0000 channel mapping (5-channel Opera Phenix)
CHANNEL_NAMES   = ['DNA', 'RNA', 'ER', 'AGP', 'Mito']
CHANNEL_MAPPING = {'DNA': 5, 'RNA': 3, 'ER': 4, 'AGP': 2, 'Mito': 1}

print(f'Downloading to:     {DOWNLOAD_DIR}')
print(f'Dataset will be at: {DATASET_DIR}')

In [ ]:
# -----------------------------------------------------------------------
# Download via jump_portrait
# -----------------------------------------------------------------------
# jump_portrait can download individual plates from S3.
# The cpg0000 A549 plates are in the 2020_11_04_CPJUMP1 batch.
# We fetch a small sample: one plate, one well, first 5 sites.
# -----------------------------------------------------------------------

from jump_portrait.fetch import get_jump_image  # pip install jump-portrait

# Plate/well/site to download (small sample for the notebook)
PLATE    = 'BR00117015'  # A549 plate from 2020_11_04_CPJUMP1
WELLS    = ['B02', 'B03', 'C02']  # 3 wells
N_SITES  = 3               # first 3 sites per well

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

downloaded = []  # (well, site, channel, path)

for well in WELLS:
    for site in range(1, N_SITES + 1):
        for channel_name, ch_idx in CHANNEL_MAPPING.items():
            out_path = DOWNLOAD_DIR / f'{well}_s{site:02d}_{channel_name}.tiff'
            if out_path.exists():
                downloaded.append((well, site, channel_name, out_path))
                continue
            try:
                img = get_jump_image(
                    source='cpg0000-jump-pilot',
                    batch='2020_11_04_CPJUMP1',
                    plate=PLATE,
                    well=well,
                    site=site,
                    channel=ch_idx,
                    cell_type=CELL_LINE,
                    correction=None,
                )
                import tifffile
                tifffile.imwrite(str(out_path), img)
                downloaded.append((well, site, channel_name, out_path))
                print(f'  Downloaded: {out_path.name}')
            except Exception as e:
                print(f'  FAILED {well} s{site} {channel_name}: {e}')

print(f'\nDownloaded {len(downloaded)} images')

## 2. Inspect a downloaded image

In [ ]:
import matplotlib.pyplot as plt
import tifffile

# Show one FOV (all 5 channels) for well B02, site 1
well, site = 'B02', 1

fig, axes = plt.subplots(1, len(CHANNEL_NAMES), figsize=(18, 4))
for ax, ch in zip(axes, CHANNEL_NAMES):
    path = DOWNLOAD_DIR / f'{well}_s{site:02d}_{ch}.tiff'
    if path.exists():
        img = tifffile.imread(str(path))
        vmax = np.percentile(img, 99.5)
        ax.imshow(img, cmap='gray', vmin=0, vmax=vmax)
        ax.set_title(f'{ch}\n{img.shape}  {img.dtype}')
    else:
        ax.set_title(f'{ch}\n(not downloaded)')
    ax.axis('off')

fig.suptitle(f'cpg0000 A549 | {well} site {site}', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Build MicroSplit training dataset

We convert the raw per-channel TIFFs into the dataset layout expected by MicroSplit:
```
training_dataset/
    combined/     float32 pixel-wise sum
    DNA/ RNA/ ER/ AGP/ Mito/   uint16 per-channel
    metadata.csv
```

In [ ]:
import sys
sys.path.insert(0, '../../../src')  # point at JUMP-MicroSplit/src

from microsplit_reproducibility.workflows.cellpainting import (
    combine_channels,
    save_dataset_images,
)
import csv

DATASET_DIR.mkdir(parents=True, exist_ok=True)

metadata_rows = []
image_id = 0

for well in WELLS:
    for site in range(1, N_SITES + 1):
        channel_images = {}
        ok = True
        for ch in CHANNEL_NAMES:
            path = DOWNLOAD_DIR / f'{well}_s{site:02d}_{ch}.tiff'
            if not path.exists():
                print(f'SKIP {well} s{site}: missing {ch}')
                ok = False
                break
            channel_images[ch] = tifffile.imread(str(path))
        if not ok:
            continue

        combined = combine_channels(channel_images, CHANNEL_NAMES)
        save_dataset_images(combined, channel_images, image_id, DATASET_DIR, CHANNEL_NAMES)

        metadata_rows.append({
            'image_id':     image_id,
            'batch':        '2020_11_04_CPJUMP1',
            'plate':        PLATE,
            'cell_type':    CELL_LINE,
            'well':         well,
            'site':         site,
            **{f'{ch}_path': f'{ch}/{image_id:06d}.tiff' for ch in CHANNEL_NAMES},
            'combined_path': f'combined/{image_id:06d}.tiff',
        })

        print(f'  image_id={image_id}: {well} site {site}')
        image_id += 1

# Write metadata.csv
meta_path = DATASET_DIR / 'metadata.csv'
with open(meta_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(metadata_rows[0].keys()))
    writer.writeheader()
    writer.writerows(metadata_rows)

print(f'\nDataset ready: {image_id} images → {DATASET_DIR}')
print(f'metadata.csv: {meta_path}')

## 4. Inspect the dataset

In [ ]:
import pandas as pd
import tifffile

meta = pd.read_csv(DATASET_DIR / 'metadata.csv')
print(f'Dataset: {len(meta)} images')
display(meta.head())

# Show combined vs individual channels for image_id=0
fig, axes = plt.subplots(1, len(CHANNEL_NAMES) + 1, figsize=(22, 4))

combined = tifffile.imread(str(DATASET_DIR / 'combined' / '000000.tiff'))
axes[0].imshow(combined, cmap='viridis', vmin=0, vmax=np.percentile(combined, 99.5))
axes[0].set_title('combined')
axes[0].axis('off')

for ax, ch in zip(axes[1:], CHANNEL_NAMES):
    img = tifffile.imread(str(DATASET_DIR / ch / '000000.tiff'))
    ax.imshow(img, cmap='gray', vmin=0, vmax=np.percentile(img, 99.5))
    ax.set_title(ch)
    ax.axis('off')

plt.suptitle('Dataset image 000000: combined + individual channels')
plt.tight_layout()
plt.show()

## Next step

Open **01_noisemodels.ipynb** to train per-channel noise models on this dataset.